# Lab

Review one service network, search its activity messages, and plan support-team coverage. Each task builds on this workflow. Use Edit and Run; include the complete relevant implementations from the lessons. Each hidden solution is standalone. Sign in before saving answers.

## 1. Discover Service Layers

```{index} 1. Discover Service Layers
```

Run BFS from HQ on the directed network. Report order, Client distance, and Remote’s unreachable marker. Explain why marking at enqueue time prevents duplicate work.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var adjacency=new Dictionary<string,string[]>{["HQ"]=new[]{"Warehouse","Helpdesk"},["Warehouse"]=new[]{"Depot"},["Helpdesk"]=new[]{"Client"},["Depot"]=new[]{"Client"},["Client"]=new[]{"Warehouse"},["Remote"]=Array.Empty<string>()};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
var adjacency=new Dictionary<string,string[]>{["HQ"]=new[]{"Warehouse","Helpdesk"},["Warehouse"]=new[]{"Depot"},["Helpdesk"]=new[]{"Client"},["Depot"]=new[]{"Client"},["Client"]=new[]{"Warehouse"},["Remote"]=Array.Empty<string>()};
var bfs=new GraphWalk(adjacency).Bfs("HQ");
Console.WriteLine($"order={string.Join(",",bfs.Order)}, Client hops={bfs.Distance["Client"]}, Remote unreachable={bfs.Distance["Remote"] is null}");
if(bfs.Order.Distinct().Count()!=bfs.Order.Length||bfs.Distance["Client"]!=2)throw new Exception("BFS contract failed.");
public sealed class GraphWalk
{
    private readonly Dictionary<string,string[]> graph;
    private readonly string[] vertices;
    public GraphWalk(Dictionary<string,string[]> adjacency)
    {
        ArgumentNullException.ThrowIfNull(adjacency);
        graph=new Dictionary<string,string[]>(StringComparer.Ordinal);
        foreach(var pair in adjacency)
        {
            if(pair.Value is null)throw new ArgumentException("Null adjacency list.");
            graph.Add(pair.Key,pair.Value.Distinct(StringComparer.Ordinal).ToArray());
        }
        foreach(var neighbors in graph.Values)
            if(neighbors.Any(v=>v is null||!graph.ContainsKey(v)))throw new ArgumentException("Unknown endpoint.");
        vertices=graph.Keys.OrderBy(v=>v,StringComparer.Ordinal).ToArray();
    }
    public sealed class Layers
    {
        public string Source { get; }
        public string[] Order { get; }
        public IReadOnlyDictionary<string,int?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        public long EdgesExamined { get; }
        internal Layers(string source,string[] order,Dictionary<string,int?> distance,Dictionary<string,string?> previous,long edges)
        {
            Source=source;Order=order;EdgesExamined=edges;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,int?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            while(at is not null)
            {
                if(path.Count>=Distance.Count)throw new InvalidOperationException("Predecessor cycle.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Invalid source path.");
            return path.ToArray();
        }
    }
    public Layers Bfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var distance=vertices.ToDictionary(v=>v,_=>(int?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new Queue<string>();var order=new List<string>();long edges=0;
        distance[source]=0;queue.Enqueue(source);
        while(queue.Count>0)
        {
            string at=queue.Dequeue();order.Add(at);trace?.Invoke($"visit {at}: layer={distance[at]}");
            foreach(string next in graph[at])
            {
                edges++;
                if(distance[next] is not null)continue;
                distance[next]=distance[at]!.Value+1;previous[next]=at;queue.Enqueue(next);
            }
        }
        return new Layers(source,order.ToArray(),distance,previous,edges);
    }
    public record Depth(string[] Preorder,string[] FinishOrder,long EdgesExamined);
    public Depth Dfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var color=vertices.ToDictionary(v=>v,_=>0,StringComparer.Ordinal);
        var stack=new Stack<(string Vertex,int Next)>();var pre=new List<string>();var finish=new List<string>();long edges=0;
        Enter(source);
        while(stack.Count>0)
        {
            var frame=stack.Pop();
            if(frame.Next==graph[frame.Vertex].Length)
            {color[frame.Vertex]=2;finish.Add(frame.Vertex);trace?.Invoke($"exit {frame.Vertex}");continue;}
            stack.Push((frame.Vertex,frame.Next+1));string next=graph[frame.Vertex][frame.Next];edges++;
            if(color[next]==0)Enter(next);
            else if(color[next]==1)trace?.Invoke($"active edge {frame.Vertex}->{next}");
        }
        return new Depth(pre.ToArray(),finish.ToArray(),edges);
        void Enter(string vertex){color[vertex]=1;pre.Add(vertex);stack.Push((vertex,0));trace?.Invoke($"enter {vertex}");}
    }
    public string[][] UndirectedComponents()
    {
        var neighbors=graph.ToDictionary(p=>p.Key,p=>p.Value.ToHashSet(StringComparer.Ordinal),StringComparer.Ordinal);
        foreach(var pair in graph)
            foreach(string next in pair.Value)
                if(!neighbors[next].Contains(pair.Key))throw new ArgumentException("Components require symmetric adjacency.");
        var seen=new HashSet<string>(StringComparer.Ordinal);var groups=new List<string[]>();
        foreach(string root in vertices)
        {
            if(!seen.Add(root))continue;
            var queue=new Queue<string>();var group=new List<string>();queue.Enqueue(root);
            while(queue.Count>0)
            {
                string at=queue.Dequeue();group.Add(at);
                foreach(string next in graph[at])if(seen.Add(next))queue.Enqueue(next);
            }
            groups.Add(group.ToArray());
        }
        return groups.ToArray();
    }
}


order=HQ,Warehouse,Helpdesk,Depot,Client, Client hops=2, Remote unreachable=True


## 2. Compare Traversal Guarantees

```{index} 2. Compare Traversal Guarantees
```

Run frame-based DFS on the same graph and compare reached vertices with BFS. Reconstruct source, Client, and Remote paths from BFS. Validate Client’s edges and length. Explain why DFS discovery order does not guarantee a minimum-hop route.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var adjacency=new Dictionary<string,string[]>{["HQ"]=new[]{"Warehouse","Helpdesk"},["Warehouse"]=new[]{"Depot"},["Helpdesk"]=new[]{"Client"},["Depot"]=new[]{"Client"},["Client"]=new[]{"Warehouse"},["Remote"]=Array.Empty<string>()};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
var adjacency=new Dictionary<string,string[]>{["HQ"]=new[]{"Warehouse","Helpdesk"},["Warehouse"]=new[]{"Depot"},["Helpdesk"]=new[]{"Client"},["Depot"]=new[]{"Client"},["Client"]=new[]{"Warehouse"},["Remote"]=Array.Empty<string>()};
var walk=new GraphWalk(adjacency);var bfs=walk.Bfs("HQ");var dfs=walk.Dfs("HQ");
if(!bfs.Order.OrderBy(x=>x,StringComparer.Ordinal).SequenceEqual(dfs.Preorder.OrderBy(x=>x,StringComparer.Ordinal)))throw new Exception("Reachability disagreement.");
var path=bfs.PathTo("Client");
if(path.Length!=bfs.Distance["Client"]+1||path[0]!="HQ"||path[^1]!="Client"||Enumerable.Range(0,path.Length-1).Any(i=>!adjacency[path[i]].Contains(path[i+1])))throw new Exception("Path invalid.");
Console.WriteLine($"DFS={string.Join(",",dfs.Preorder)}, Client path={string.Join("->",path)}");
Console.WriteLine($"source length={bfs.PathTo("HQ").Length}, Remote length={bfs.PathTo("Remote").Length}");
public sealed class GraphWalk
{
    private readonly Dictionary<string,string[]> graph;
    private readonly string[] vertices;
    public GraphWalk(Dictionary<string,string[]> adjacency)
    {
        ArgumentNullException.ThrowIfNull(adjacency);
        graph=new Dictionary<string,string[]>(StringComparer.Ordinal);
        foreach(var pair in adjacency)
        {
            if(pair.Value is null)throw new ArgumentException("Null adjacency list.");
            graph.Add(pair.Key,pair.Value.Distinct(StringComparer.Ordinal).ToArray());
        }
        foreach(var neighbors in graph.Values)
            if(neighbors.Any(v=>v is null||!graph.ContainsKey(v)))throw new ArgumentException("Unknown endpoint.");
        vertices=graph.Keys.OrderBy(v=>v,StringComparer.Ordinal).ToArray();
    }
    public sealed class Layers
    {
        public string Source { get; }
        public string[] Order { get; }
        public IReadOnlyDictionary<string,int?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        public long EdgesExamined { get; }
        internal Layers(string source,string[] order,Dictionary<string,int?> distance,Dictionary<string,string?> previous,long edges)
        {
            Source=source;Order=order;EdgesExamined=edges;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,int?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            while(at is not null)
            {
                if(path.Count>=Distance.Count)throw new InvalidOperationException("Predecessor cycle.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Invalid source path.");
            return path.ToArray();
        }
    }
    public Layers Bfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var distance=vertices.ToDictionary(v=>v,_=>(int?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new Queue<string>();var order=new List<string>();long edges=0;
        distance[source]=0;queue.Enqueue(source);
        while(queue.Count>0)
        {
            string at=queue.Dequeue();order.Add(at);trace?.Invoke($"visit {at}: layer={distance[at]}");
            foreach(string next in graph[at])
            {
                edges++;
                if(distance[next] is not null)continue;
                distance[next]=distance[at]!.Value+1;previous[next]=at;queue.Enqueue(next);
            }
        }
        return new Layers(source,order.ToArray(),distance,previous,edges);
    }
    public record Depth(string[] Preorder,string[] FinishOrder,long EdgesExamined);
    public Depth Dfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var color=vertices.ToDictionary(v=>v,_=>0,StringComparer.Ordinal);
        var stack=new Stack<(string Vertex,int Next)>();var pre=new List<string>();var finish=new List<string>();long edges=0;
        Enter(source);
        while(stack.Count>0)
        {
            var frame=stack.Pop();
            if(frame.Next==graph[frame.Vertex].Length)
            {color[frame.Vertex]=2;finish.Add(frame.Vertex);trace?.Invoke($"exit {frame.Vertex}");continue;}
            stack.Push((frame.Vertex,frame.Next+1));string next=graph[frame.Vertex][frame.Next];edges++;
            if(color[next]==0)Enter(next);
            else if(color[next]==1)trace?.Invoke($"active edge {frame.Vertex}->{next}");
        }
        return new Depth(pre.ToArray(),finish.ToArray(),edges);
        void Enter(string vertex){color[vertex]=1;pre.Add(vertex);stack.Push((vertex,0));trace?.Invoke($"enter {vertex}");}
    }
    public string[][] UndirectedComponents()
    {
        var neighbors=graph.ToDictionary(p=>p.Key,p=>p.Value.ToHashSet(StringComparer.Ordinal),StringComparer.Ordinal);
        foreach(var pair in graph)
            foreach(string next in pair.Value)
                if(!neighbors[next].Contains(pair.Key))throw new ArgumentException("Components require symmetric adjacency.");
        var seen=new HashSet<string>(StringComparer.Ordinal);var groups=new List<string[]>();
        foreach(string root in vertices)
        {
            if(!seen.Add(root))continue;
            var queue=new Queue<string>();var group=new List<string>();queue.Enqueue(root);
            while(queue.Count>0)
            {
                string at=queue.Dequeue();group.Add(at);
                foreach(string next in graph[at])if(seen.Add(next))queue.Enqueue(next);
            }
            groups.Add(group.ToArray());
        }
        return groups.ToArray();
    }
}


DFS=HQ,Warehouse,Depot,Client,Helpdesk, Client path=HQ->Helpdesk->Client
source length=1, Remote length=0


## 3. Search an Activity Message

```{index} 3. Search an Activity Message
```

Search a service message for every overlapping occurrence of Alert. Check direct and prepared-prefix outputs agree. Also search for lowercase alert and explain the ordinal policy.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

string message="Alert AlertAlert";string pattern="Alert";
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
string message="Alert AlertAlert";string pattern="Alert";
var direct=PatternScan.Direct(message,pattern);var prepared=new PatternScan(pattern);var kmp=prepared.Search(message);
if(!direct.Positions.SequenceEqual(kmp.Positions))throw new Exception("Message search disagreement.");
Console.WriteLine($"Alert positions={string.Join(",",kmp.Positions)}, lowercase matches={new PatternScan("alert").Search(message).Positions.Length}");
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


Alert positions=0,6,11, lowercase matches=0


## 4. Count Reused Preparation

```{index} 4. Count Reused Preparation
```

Search two repetitive activity messages with the same pattern aaaaab. Count direct comparisons, prepare KMP once, and count both scans plus preparation. Check identical matches; do not call these counts elapsed time.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

string[] messages={new string('a',60)+"b",new string('a',100)+"b"};string pattern="aaaaab";
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
string[] messages={new string('a',60)+"b",new string('a',100)+"b"};string pattern="aaaaab";
var prepared=new PatternScan(pattern);long directWork=0,scanWork=0;
foreach(string message in messages)
{
 var d=PatternScan.Direct(message,pattern);var k=prepared.Search(message);
 if(!d.Positions.SequenceEqual(k.Positions))throw new Exception("Workload outputs differ.");
 directWork+=d.Comparisons;scanWork+=k.Comparisons;
}
Console.WriteLine($"direct comparisons={directWork}, KMP prep={prepared.PreparationComparisons}, scans={scanWork}, total={prepared.PreparationComparisons+scanWork}");
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


direct comparisons=912, KMP prep=9, scans=312, total=321


## 5. Recommend a Coverage Strategy

```{index} 5. Recommend a Coverage Strategy
```

The service manager needs all six skills with as few teams as possible. Validate greedy and exact coverage, compare team counts, and state when an exact optimum versus a feasible heuristic result satisfies the requirement. Report the candidate-set growth for 20 teams.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

int[] teams={0b001111,0b010011,0b101100};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
int[] teams={0b001111,0b010011,0b101100};
var g=TeamCover.Greedy(6,teams);var e=TeamCover.Exact(6,teams)!;
foreach(var plan in new[]{g,e})
 if(plan.Indices.Distinct().Count()!=plan.Indices.Length||plan.Indices.Aggregate(0,(mask,i)=>mask|teams[i])!=63)throw new Exception("Coverage witness invalid.");
Console.WriteLine($"greedy count={g.Indices.Length}, exact count={e.Indices.Length}, both cover all six skills");
Console.WriteLine($"20 teams have {BigInteger.One<<20} candidate subsets; exact teaching limit is 20 teams");
Console.WriteLine("Require exact count when minimum is mandatory; a feasible heuristic needs a separately accepted quality policy.");
public static class TeamCover
{
    public record Plan(int[] Indices,int Covered);
    // Teams are positions; each mask identifies covered requirements, up to 20.
    public static Plan Greedy(int requirements,int[] teams)
    {
        int full=Validate(requirements,teams);int covered=0;var chosen=new List<int>();
        while(covered!=full)
        {
            int best=-1,gain=0;
            for(int i=0;i<teams.Length;i++)
            {int next=System.Numerics.BitOperations.PopCount((uint)(teams[i]&~covered));if(next>gain){best=i;gain=next;}}
            if(best<0)break;
            chosen.Add(best);covered|=teams[best];
        }
        return new Plan(chosen.ToArray(),covered);
    }
    public static Plan? Exact(int requirements,int[] teams)
    {
        int full=Validate(requirements,teams);
        if(teams.Length>20)throw new ArgumentException("Exact teaching search is capped at 20 teams.");
        int bestCount=int.MaxValue,bestMask=0;
        for(int mask=0;mask<(1<<teams.Length);mask++)
        {
            int count=System.Numerics.BitOperations.PopCount((uint)mask);
            if(count>=bestCount)continue;
            int covered=0;for(int i=0;i<teams.Length;i++)if((mask&(1<<i))!=0)covered|=teams[i];
            if(covered==full){bestCount=count;bestMask=mask;}
        }
        return bestCount==int.MaxValue?null:new Plan(Enumerable.Range(0,teams.Length).Where(i=>(bestMask&(1<<i))!=0).ToArray(),full);
    }
    private static int Validate(int requirements,int[] teams)
    {
        ArgumentNullException.ThrowIfNull(teams);
        if(requirements<0||requirements>20)throw new ArgumentOutOfRangeException(nameof(requirements));
        int full=(1<<requirements)-1;
        if(teams.Any(mask=>mask<0||(mask&~full)!=0))throw new ArgumentException("Unknown requirement bit.");
        return full;
    }
}


greedy count=3, exact count=2, both cover all six skills
20 teams have 1048576 candidate subsets; exact teaching limit is 20 teams
Require exact count when minimum is mandatory; a feasible heuristic needs a separately accepted quality policy.


## Project Connection

```{index} Project Connection
```

Compare two strategies for one project feature under an identical contract. Include checked outputs, a complete workload including preparation, time/space limits, and a recommendation tied to the required guarantee.

## Submit

```{index} Submit
```

Submit code, verified outputs, and explanations through the assignment panel when available. Durable capstone work follows the existing VS Code/GitHub project track.